# GridPulse BR — Annual Regulatory Compliance

## Objective

Build an annual regulatory compliance product by comparing actual annual DEC
and FEC values against the corresponding ANEEL annual regulatory limits.

## Business grain

One row represents:

- one consumer-unit set
- one year

## Important

The regulatory limits are annual. Therefore, monthly actual DEC/FEC values
must first be aggregated to the annual level before regulatory comparison.

In [0]:
from pyspark.sql import functions as F

ACTUAL_TABLE = "workspace.gridpulse.gold_service_quality"
LIMIT_TABLE = "workspace.gridpulse.silver_aneel_regulatory_limits"
TARGET_TABLE = "workspace.gridpulse.gold_annual_regulatory_compliance"

print(f"Actual table : {ACTUAL_TABLE}")
print(f"Limit table  : {LIMIT_TABLE}")
print(f"Target table : {TARGET_TABLE}")

In [0]:
df_actual = spark.table(ACTUAL_TABLE)
df_limits = spark.table(LIMIT_TABLE)

print(f"Actual monthly rows : {df_actual.count():,}")
print(f"Limit rows          : {df_limits.count():,}")

In [0]:
df_actual_annual = (
    df_actual
    .groupBy(
        "IdeConjUndConsumidoras",
        "DscConjUndConsumidoras",
        "SigAgente",
        "NumCNPJ",
        "AnoIndice"
    )
    .agg(
        F.sum("DEC").alias("DEC_actual"),
        F.sum("FEC").alias("FEC_actual"),
        F.countDistinct("NumPeriodoIndice").alias("months_available")
    )
)

display(
    df_actual_annual
    .groupBy("AnoIndice")
    .agg(
        F.count("*").alias("rows"),
        F.min("months_available").alias("min_months"),
        F.max("months_available").alias("max_months")
    )
    .orderBy("AnoIndice")
)

## 2. Classify annual data completeness

Regulatory compliance is evaluated only when all twelve monthly observations
are available for a consumer-unit set and year.

Partial annual observations are preserved in the Gold product but are not
classified as compliant or non-compliant.

In [0]:
df_actual_annual = (
    df_actual_annual
    .withColumn(
        "year_status",
        F.when(
            F.col("months_available") == 12,
            F.lit("COMPLETE")
        ).otherwise(F.lit("PARTIAL"))
    )
)

display(
    df_actual_annual
    .groupBy("AnoIndice", "year_status")
    .count()
    .orderBy("AnoIndice", "year_status")
)

## 3. Prepare annual regulatory limits

In [0]:
df_limits_wide = (
    df_limits
    .filter(F.col("SigIndicador").isin("DEC", "FEC"))
    .groupBy(
        "IdeConjUndConsumidoras",
        "AnoLimiteQualidade"
    )
    .pivot("SigIndicador", ["DEC", "FEC"])
    .agg(F.first("VlrLimite"))
    .withColumnRenamed("DEC", "DEC_limit")
    .withColumnRenamed("FEC", "FEC_limit")
)

print(f"Annual limit rows: {df_limits_wide.count():,}")

display(df_limits_wide.limit(10))

In [0]:
limit_duplicates = (
    df_limits_wide
    .groupBy(
        "IdeConjUndConsumidoras",
        "AnoLimiteQualidade"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate annual limit keys: {limit_duplicates:,}")

assert limit_duplicates == 0

print("Annual limit grain validation passed.")

## 4. Join annual actuals with regulatory limits

In [0]:
df_compliance = (
    df_actual_annual.alias("a")
    .join(
        df_limits_wide.alias("l"),
        (
            (F.col("a.IdeConjUndConsumidoras") ==
             F.col("l.IdeConjUndConsumidoras"))
            &
            (F.col("a.AnoIndice") ==
             F.col("l.AnoLimiteQualidade"))
        ),
        "left"
    )
    .select(
        F.col("a.IdeConjUndConsumidoras"),
        F.col("a.DscConjUndConsumidoras"),
        F.col("a.SigAgente"),
        F.col("a.NumCNPJ"),
        F.col("a.AnoIndice"),
        F.col("a.months_available"),
        F.col("a.year_status"),
        F.col("a.DEC_actual"),
        F.col("a.FEC_actual"),
        F.col("l.DEC_limit"),
        F.col("l.FEC_limit")
    )
)

In [0]:
df_compliance = (
    df_compliance
    .withColumn(
        "DEC_status",
        F.when(
            F.col("year_status") == "PARTIAL",
            "PARTIAL_YEAR"
        )
        .when(
            F.col("DEC_limit").isNull(),
            "NO_LIMIT"
        )
        .when(
            F.col("DEC_actual") <= F.col("DEC_limit"),
            "COMPLIANT"
        )
        .otherwise("NON_COMPLIANT")
    )
    .withColumn(
        "FEC_status",
        F.when(
            F.col("year_status") == "PARTIAL",
            "PARTIAL_YEAR"
        )
        .when(
            F.col("FEC_limit").isNull(),
            "NO_LIMIT"
        )
        .when(
            F.col("FEC_actual") <= F.col("FEC_limit"),
            "COMPLIANT"
        )
        .otherwise("NON_COMPLIANT")
    )
    .withColumn(
        "DEC_variance",
        F.col("DEC_actual") - F.col("DEC_limit")
    )
    .withColumn(
        "FEC_variance",
        F.col("FEC_actual") - F.col("FEC_limit")
    )
)

In [0]:
display(
    df_compliance
    .groupBy(
        "AnoIndice",
        "year_status",
        "DEC_status"
    )
    .count()
    .orderBy(
        "AnoIndice",
        "year_status",
        "DEC_status"
    )
)

In [0]:
display(
    df_compliance
    .groupBy(
        "AnoIndice",
        "year_status",
        "FEC_status"
    )
    .count()
    .orderBy(
        "AnoIndice",
        "year_status",
        "FEC_status"
    )
)

## 4. Join annual actuals with regulatory limits

Join annual DEC/FEC actual performance to the corresponding ANEEL regulatory
limits using consumer-unit set and year.

The join preserves annual actual observations even when a corresponding
regulatory limit is unavailable.

In [0]:
df_compliance = (
    df_actual_annual.alias("a")
    .join(
        df_limits_wide.alias("l"),
        (
            (F.col("a.IdeConjUndConsumidoras") ==
             F.col("l.IdeConjUndConsumidoras"))
            &
            (F.col("a.AnoIndice") ==
             F.col("l.AnoLimiteQualidade"))
        ),
        how="left"
    )
    .select(
        F.col("a.IdeConjUndConsumidoras"),
        F.col("a.DscConjUndConsumidoras"),
        F.col("a.SigAgente"),
        F.col("a.NumCNPJ"),
        F.col("a.AnoIndice"),
        F.col("a.months_available"),
        F.col("a.year_status"),
        F.col("a.DEC_actual"),
        F.col("a.FEC_actual"),
        F.col("l.DEC_limit"),
        F.col("l.FEC_limit")
    )
)

print(f"Annual actual rows : {df_actual_annual.count():,}")
print(f"Joined rows        : {df_compliance.count():,}")

In [0]:
actual_annual_count = df_actual_annual.count()
joined_count = df_compliance.count()

assert joined_count == actual_annual_count, (
    f"Join reconciliation failed: expected "
    f"{actual_annual_count:,}, got {joined_count:,}"
)

print("Actual-to-limit join reconciliation passed.")

## 5. Calculate regulatory compliance

Only complete annual observations are eligible for compliance classification.

Status rules:

- `PARTIAL_YEAR`: fewer than 12 monthly observations
- `NO_LIMIT`: complete year without a corresponding regulatory limit
- `COMPLIANT`: annual actual value is less than or equal to the limit
- `NON_COMPLIANT`: annual actual value exceeds the regulatory limit

In [0]:
df_compliance = (
    df_compliance

    .withColumn(
        "DEC_variance",
        F.when(
            F.col("DEC_limit").isNotNull(),
            F.col("DEC_actual") - F.col("DEC_limit")
        )
    )

    .withColumn(
        "FEC_variance",
        F.when(
            F.col("FEC_limit").isNotNull(),
            F.col("FEC_actual") - F.col("FEC_limit")
        )
    )

    .withColumn(
        "DEC_status",
        F.when(
            F.col("year_status") == "PARTIAL",
            "PARTIAL_YEAR"
        )
        .when(
            F.col("DEC_limit").isNull(),
            "NO_LIMIT"
        )
        .when(
            F.col("DEC_actual") <= F.col("DEC_limit"),
            "COMPLIANT"
        )
        .otherwise("NON_COMPLIANT")
    )

    .withColumn(
        "FEC_status",
        F.when(
            F.col("year_status") == "PARTIAL",
            "PARTIAL_YEAR"
        )
        .when(
            F.col("FEC_limit").isNull(),
            "NO_LIMIT"
        )
        .when(
            F.col("FEC_actual") <= F.col("FEC_limit"),
            "COMPLIANT"
        )
        .otherwise("NON_COMPLIANT")
    )
)

In [0]:
display(
    df_compliance
    .groupBy(
        "AnoIndice",
        "DEC_status"
    )
    .count()
    .orderBy(
        "AnoIndice",
        "DEC_status"
    )
)

In [0]:
display(
    df_compliance
    .groupBy(
        "AnoIndice",
        "FEC_status"
    )
    .count()
    .orderBy(
        "AnoIndice",
        "FEC_status"
    )
)

In [0]:
display(
    df_compliance
    .filter(F.col("AnoIndice") == 2026)
    .groupBy(
        "year_status",
        "DEC_status",
        "FEC_status"
    )
    .count()
)

## 6. Validate final analytical grain

The final Gold product must contain one row per consumer-unit set and year.

In [0]:
GOLD_KEY = [
    "IdeConjUndConsumidoras",
    "AnoIndice"
]

duplicate_gold = (
    df_compliance
    .groupBy(*GOLD_KEY)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_gold_count = duplicate_gold.count()

print(f"Duplicate Gold keys: {duplicate_gold_count:,}")

assert duplicate_gold_count == 0, (
    f"Gold grain validation failed: "
    f"{duplicate_gold_count:,} duplicate keys found."
)

print("Annual regulatory Gold grain validation passed.")

## 7. Persist annual regulatory compliance Gold

In [0]:
(
    df_compliance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Gold regulatory compliance table created: {TARGET_TABLE}")

In [0]:
df_gold_persisted = spark.table(TARGET_TABLE)

persisted_gold_count = df_gold_persisted.count()
expected_gold_count = df_compliance.count()

print(f"Expected Gold rows : {expected_gold_count:,}")
print(f"Actual Gold rows   : {persisted_gold_count:,}")

assert persisted_gold_count == expected_gold_count

print("Annual regulatory compliance Gold reconciliation passed.")

## 8. Product summary

In [0]:
summary = (
    df_compliance
    .groupBy("AnoIndice")
    .agg(
        F.count("*").alias("consumer_sets"),
        F.sum(
            F.when(F.col("year_status") == "COMPLETE", 1).otherwise(0)
        ).alias("complete_years"),
        F.sum(
            F.when(F.col("DEC_status") == "COMPLIANT", 1).otherwise(0)
        ).alias("dec_compliant"),
        F.sum(
            F.when(F.col("DEC_status") == "NON_COMPLIANT", 1).otherwise(0)
        ).alias("dec_non_compliant"),
        F.sum(
            F.when(F.col("FEC_status") == "COMPLIANT", 1).otherwise(0)
        ).alias("fec_compliant"),
        F.sum(
            F.when(F.col("FEC_status") == "NON_COMPLIANT", 1).otherwise(0)
        ).alias("fec_non_compliant")
    )
    .orderBy("AnoIndice")
)

display(summary)